In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 1.7 The Falling Chain

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume I — Elementary Mechanics",
    number="1.7",
    title="The Falling Chain",
    blurb="Three chains, three surprises: a chain pouring off a table, a "
    "chain piling on the floor with three times its weight, and a "
    "folded chain that falls faster than gravity. Variable-mass "
    "mechanics and where energy quietly disappears.",
    difficulty="advanced",
    estimate="90–120 min",
)

## Notebook overview

A chain is the simplest system in which **mass is set into motion (or stopped) a
bit at a time**, and that single complication overturns the reflex that "it's
just gravity." We take one idealised chain (inextensible, perfectly flexible,
linear density $\lambda$) through three classic situations, and in each the
naive answer is wrong in a different way:

1. **Act I: a chain sliding off a table.** The gentle case: nothing changes
   speed abruptly, energy is conserved, and the overhang grows like $\cosh$.
2. **Act II: a chain piling on the floor.** Landing links must be *stopped*, and
   the floor reads **three times** the weight already landed.
3. **Act III: a folded chain, one end released.** The subtle case: the free end
   falls *faster than $g$*, and the two textbook analyses (energy versus
   momentum) disagree about where the energy goes.

The unifying tool is **Newton's law in its general form**, $\mathbf F_\text{ext}
= d\mathbf p/dt$ for the *whole* system, applied with care to the momentum
carried by mass entering or leaving the moving part. This is the classical
cousin of the rocket equation, and most "chain paradoxes" are just $F=ma$ used
where it does not apply.

> **How to read the checks.** Each exercise ends with a `validate` call against
> an independent fact: a closed-form solution, a conserved quantity, an exact
> ratio. A ✓ is strong evidence we got it right; a ✗ is a prompt to *locate the
> discrepancy* (a sign, a missing momentum-flux term, too tight a tolerance), not
> a verdict.

> **Scope.** A working review, not a treatise on variable-mass dynamics. For the
> underlying mechanics see Nolting, *Theoretische Physik 1* {cite}`nolting1`; the
> chain problems are Cambridge/Irodov staples, and the folded chain in
> particular has a careful modern literature (Calkin & March) that we follow.

## Theory in brief

### Newton's law when the moving mass changes

For a body of *fixed* mass, $F=ma$. When mass is continuously added to or removed
from the moving part, that is no longer the right statement; the safe law is
Newton's original one, applied to a *fixed collection of matter* (a closed
system):

```{math}
:label: eq-dpdt
\mathbf F_\text{ext} = \frac{d\mathbf p}{dt},
```

where $\mathbf p$ is the total momentum of *all* the matter and
$\mathbf F_\text{ext}$ the total external force. The trap is to write
$F = \frac{d}{dt}(mv) = \dot m v + m\dot v$ for the *moving piece alone* and
forget that the mass crossing the boundary also carries momentum. Done correctly
(accounting for the momentum flux of the mass entering or leaving) {eq}`eq-dpdt`
is unambiguous. (Each act below applies it to its own boundary.)

### Energy is not guaranteed

Momentum/force bookkeeping is always valid. **Energy conservation is not.** When
a link is jerked from rest to speed $v$ (or from $v$ to rest) over an
infinitesimal distance, that is an **inelastic** event (like a perfectly
inelastic collision), and mechanical energy can be lost to it:

```{math}
:label: eq-energy-loss
\frac{dE_\text{mech}}{dt} \le 0 \quad\text{when links change speed abruptly.}
```

So the reliable strategy is always force/momentum first; then ask, separately,
whether energy happened to be conserved. Act I conserves it (nothing jerks); Act
II plainly dissipates it (links slam to rest); Act III is the subtle one, where
the answer depends on how the chain negotiates its fold, and that is the whole
lesson. The per-act theory is developed in each act's statement below.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar

from ecp import draw, validate
from ecp.animate import show

LAM = 1.0  # linear mass density  λ  [kg/m]
L = 1.0  # chain length  L  [m]
G = 9.81  # gravitational acceleration  g  [m/s²]

---
## Act I — A chain sliding off a table edge

**The setup and its theory.** A uniform chain of length $L$ and linear density
$\lambda$ lies on a frictionless table; a short piece of length $x$ hangs over
the edge ({numref}`fig-chain-table`). Only the hanging piece feels a net
pull (its weight $\lambda x g$), but it drags the *whole* chain of mass
$\lambda L$ with it (the chain is inextensible, so every link moves at the same
speed $\dot x$). No link changes speed abruptly here, so {eq}`eq-dpdt` reduces to
ordinary $F=ma$ on the whole chain:

```{math}
:label: eq-table
(\lambda L)\,\ddot x = (\lambda x)\,g \qquad\Longrightarrow\qquad
\ddot x = \frac{g}{L}\,x, \qquad 0 < x < L.
```

This is the *unstable* linear equation $\ddot x = +\omega^2 x$ with
$\omega=\sqrt{g/L}$ (exponential growth, not oscillation), so from rest at $x_0$
it solves to $x(t) = x_0\cosh(\sqrt{g/L}\,t)$. Because nothing is jerked, energy
is conserved; this is the calibration case where energy and momentum agree.

**This exercise.**

1. Implement {eq}`eq-table` and integrate from a small overhang with
   `scipy.integrate.solve_ivp` (default `RK45`, `rtol=1e-11`, `atol=1e-12`, a
   terminal `leave_table` event, dense `t_eval`), plotting $x(t)$.
2. Compare to the closed form $x_0\cosh(\sqrt{g/L}\,t)$.
3. Evaluate the total energy $E=\tfrac12\lambda L\dot x^2-\tfrac12\lambda g x^2$
   along the solution and confirm its relative drift stays at the solver
   tolerance: energy is conserved while the chain slides.

In [ ]:
# (solution hidden on the public site)


### Solution — Act I

In [ ]:
# (solution hidden on the public site)


### Validation — Act I

In [ ]:
validate.close(x1, x_closed, "the table chain follows x₀cosh(√(g/L)t)", rtol=1e-6)
validate.conserved(
    E1, "energy is conserved while the chain slides (no dissipation)", rel_drift=1e-6
)

---
## Act II — A chain falling onto the floor

**The setup and its theory.** Now hold the chain vertically with its lower end
just touching a floor (or scale) and release it ({numref}`fig-chain-floor`). The
chain falls freely; as it does, its bottom links reach the floor and **stop
dead**. What does the floor read when a length $y$ has piled up?

Apply {eq}`eq-dpdt` to the floor's contact force. It has two parts. First, the
**static weight** of the chain already at rest on the floor, $\lambda y g$.
Second, the force to *stop the arriving links*: in a time $dt$ a length
$v\,dt$ lands, i.e. mass $dm = \lambda v\,dt$ arriving at speed $v$, so the floor
must destroy momentum at the rate $\dfrac{dm}{dt}\,v = \lambda v^2$. The links
arrive in free fall, $v^2 = 2gy$, hence

```{math}
:label: eq-floor
F = \underbrace{\lambda y g}_{\text{weight landed}} + \underbrace{\lambda v^2}_{\text{stopping force}}
  = \lambda y g + 2\lambda y g = 3\,\lambda y g,
```

**three times** the weight of the chain on the floor. Stopping mass is not free.

**This exercise.**

1. Implement the two contributions and the total $F(y)$ on a grid of landed
   lengths (`numpy.linspace`).
2. Show the ratio $F/(\lambda y g)$ equals exactly $3$ throughout the fall.
3. Plot the force against the naive expectation (just the landed weight).

In [ ]:
# (solution hidden on the public site)


### Solution — Act II

In [ ]:
# (solution hidden on the public site)


We caption the comparison as a numbered figure, since it carries the headline
factor-of-three result.

In [ ]:
# (solution hidden on the public site)


### Validation — Act II

In [ ]:
validate.close(
    ratio,
    3.0 * np.ones_like(ratio),
    "the floor force is three times the landed weight",
    rtol=1e-6,
)

---
## Act III — The folded falling chain

**The setup and its theory (the subtle one).** Fold the chain in two and hang it
from the ceiling: one end is **fixed**, the other (the **free** end) starts
beside it at the top and is released ({numref}`fig-chain-folded`). As the free
end falls a distance $x$, links peel off the moving strand and join the hanging
strand at the **fold**. Geometry: with both strands measured from the ceiling,
the fold sits at depth $(L+x)/2$, so the moving strand has length $(L-x)/2$ and
*shrinks* as $x$ grows.

Here the two textbook analyses **disagree**, and that disagreement is the point.

*Energy-conserving chain.* If the fold is an ideal frictionless bend that does no
net work destroying motion (the modern, experimentally supported picture,
Calkin & March), mechanical energy is conserved. Working out $T+V=\text{const}$
for the shrinking moving strand gives the free-end speed

```{math}
:label: eq-folded
v^2(x) = \frac{g\,x\,(2L - x)}{L - x},
\qquad a(x) = \tfrac12\frac{dv^2}{dx} = \frac{g\,(2L^2 - 2Lx + x^2)}{2\,(L-x)^2}.
```

Since $v^2(x) > 2gx$ for all $x>0$, **the free end falls faster than free fall**,
and $a(x) \ge g$ rising to $\infty$ as $x\to L$.

*Inelastic-fold alternative.* If instead each link is stopped dead at the fold
(treating it like the floor of Act II), no force drives the moving strand beyond
gravity: it is in pure **free fall**, $v^2 = 2gx$, $a=g$, and mechanical energy is
**dissipated** at the fold per {eq}`eq-energy-loss`. The gap between the two
predictions is exactly the energy that this model loses.

Experiment (a real chain) follows the *energy-conserving* branch: it does fall
faster than $g$. (Note: some older texts assert the folded chain loses energy by
analogy with Act II; that inelastic model is the *slower*, free-fall one, so it
cannot be the faster-than-$g$ motion that is actually observed. We take the
honest, Calkin–March view and contrast the two.)

**This exercise.**

1. Write the closed-form energy-conserving $v^2(x)$ and integrate the
   momentum/force EOM $\ddot x = a(x)$ with `scipy.integrate.solve_ivp`
   (a terminal event short of the $x\to L$ singularity); confirm they agree.
2. Show the free end outruns free fall ($v^2 > 2gx$).
3. Quantify the energy gap to the inelastic-fold model and discuss the
   dissipation.

In [ ]:
# (solution hidden on the public site)


### Solution — Act III

In [ ]:
# (solution hidden on the public site)


### Validation — Act III (parts 1, 2)

In [ ]:
validate.close(
    v3**2,
    v2_closed,
    "the integrated EOM matches the energy-conserving v²(x)",
    rtol=1e-6,
)
validate.check(
    np.all(v3[1:] ** 2 > 2 * G * x3[1:]),
    "the folded chain's free end falls faster than free fall",
    f"v²/2gx in [{ratio_ff.min():.2f}, {ratio_ff.max():.2f}]",
)

**Part 3 — the energy gap.** The energy-conserving chain keeps $E=E_0$. The
inelastic-fold model instead lets the moving strand free-fall ($v^2=2gx$) and
*dissipates* the difference. Comparing the kinetic energies of the two models at
the same configuration gives a clean closed form for the gap,
$\Delta E(x) = \tfrac14\lambda g x^2 > 0$: the energy the inelastic fold would
destroy, and exactly what the energy-conserving fold instead supplies as work.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    np.all(energy_gap[1:] > 0),
    "the folded-chain idealisations disagree — energy is lost at an inelastic fold",
    f"gap(x≈0.9L) = {energy_gap[-1]:.4f} J",
)
validate.close(
    energy_gap,
    0.25 * LAM * G * x3**2,
    "the energy gap is exactly ¼λgx²",
    rtol=1e-6,
    atol=1e-9,
)

## Exercise 4 — Animate the folded chain beside a free-fall mass (worked)

Motion is the whole point here, so this one is genuinely an animation: watch the
folded chain fall while a mass dropped at the same instant free-falls beside it,
and *see* the chain's free end pull ahead. The validation checks the **physics of
the animated data** (the free end's lead over the reference), not the drawing.

In [ ]:
# (solution hidden on the public site)


### Validation — Exercise 4

At every instant the chain's free end has fallen farther than the free-fall mass.

In [ ]:
lead = xA[1:] - np.minimum(0.5 * G * tA[1:] ** 2, L)
validate.check(
    np.all(lead > 0),
    "the chain end stays ahead of the free-fall reference",
    f"min lead = {lead.min():.4f} m, max lead = {lead.max():.3f} m",
)

## Exercise 5 — The free-end acceleration (student exercise)

No animation here: the lesson is a *curve*, not a motion. Using the
acceleration $a(x)$ of {eq}`eq-folded`, show quantitatively that the free end
accelerates faster than gravity everywhere and that the acceleration runs away as
the chain straightens.

1. Evaluate $a(x)$ on a grid $x\in(0, 0.95L)$ (`numpy.linspace` with the
   `a_folded` helper) and plot $a/g$ versus $x/L$.
2. Confirm with `numpy.all` that $a > g$ throughout, and that $a\to\infty$ as
   $x\to L$. Relate the runaway to the energy the fold must supply ever faster
   as the moving strand shortens.

A ✗ points at the $a(x)$ expression, not at any drawing.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (a_folded() from earlier in this notebook — reused unchanged)
validate.check(
    np.all(a_folded(xg) > G),
    "the free-end acceleration exceeds g everywhere",
    f"min a/g = {a_over_g.min():.3f}",
)

## Exercise 6 — Act IV: the chain in reverse, or the rocket equation

Every act so far *gathered* mass; the rocket *sheds* it, and the same
{eq}`eq-dpdt` bookkeeping — momentum carried across a moving boundary,
never $F = ma$ on a quietly changing mass — delivers the most
consequential formula in astronautics. Ejecting propellant at exhaust
speed $v_e$ (relative to the rocket) at rate $\mu = -dm/dt$ gives
$m\,\dot v = v_e \mu$, and integrating from mass $m_0$ to $m_1$,

```{math}
:label: eq-rocket
\Delta v \;=\; v_e \ln\frac{m_0}{m_1}
```

— Tsiolkovsky's equation, with its tyrannical logarithm: a mass ratio
of $10$ buys only $2.3\,v_e$.

1. Integrate $\dot v = v_e\mu/m(t)$ with `scipy.integrate.solve_ivp`
   ($v_e = 3\ \mathrm{km\,s^{-1}}$, $m_0/m_1 = 10$, constant
   $\mu$) and verify {eq}`eq-rocket` to `rtol=1e-9`.
2. Launch it vertically: add $-g$ to the integrand and verify the
   closed form $v = v_e\ln(m_0/m_1) - g\,t_{\rm burn}$ to the same
   tolerance. The **gravity loss** $g\,t_{\rm burn} = 1766\ \mathrm{
   m\,s^{-1}}$ eats a quarter of the free-space budget: every second
   spent burning is a second spent hovering, which is why real rockets
   throttle up and pitch over early.
3. Why rockets have stages: give the rocket a structure fraction
   $s = 0.10$ (tanks and engines are a tenth of each stage's mass —
   they cannot be burned) and a payload of $1\%$. Verify the
   single-stage ceiling: even with *zero* payload,
   $\Delta v \le v_e \ln(1/s) = 6908\ \mathrm{m\,s^{-1}}$, and
   with the payload it reaches only $6649\ \mathrm{m\,s^{-1}}$
   (`rtol=1e-2`) — short of the $\approx 9\ \mathrm{km\,s^{-1}}$
   that reaching orbit costs. Then split the same total mass into two
   stages, optimize the split with `scipy.optimize.minimize_scalar`,
   and verify $\Delta v = 9964\ \mathrm{m\,s^{-1}}$ (`rtol=1e-2`):
   dropping the first stage's dead structure mid-flight beats any
   single stage built from identical parts. Orbit is unreachable in
   one piece and comfortable in two — the entire architecture of
   spaceflight, from one logarithm.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    gap_free < 1e-9 and gap_grav < 1e-9,
    "the integrated variable-mass dynamics lands on Tsiolkovsky's "
    "logarithm — and on its gravity-loss correction — to nine digits",
    f"relative gaps {gap_free:.1e}, {gap_grav:.1e}",
)
validate.close(
    np.array([G * T_BURN / dv_tsiol]),
    np.array([0.2557]),
    "the vertical burn spends a quarter of its budget hovering: "
    "gravity loss is why rockets pitch over early",
    rtol=1e-2,
)
validate.check(
    dv_single < 9000.0 < dv_staged and dv_ceiling < 9000.0,
    "with 10% structure, orbit (~9 km/s) is unreachable in one piece "
    "even with zero payload — and comfortable in two: why rockets "
    "have stages",
    f"single {dv_single:.0f} (ceiling {dv_ceiling:.0f}) vs staged "
    f"{dv_staged:.0f} m/s",
)
validate.close(
    np.array([dv_single, dv_staged]),
    np.array([6649.0, 9964.0]),
    "with both architectures' budgets pinned to the metre per second",
    rtol=1e-2,
)

## Notebook summary

Three chains, three failures of "just gravity":

- **Act I:** the sliding chain accelerates as $\ddot x=(g/L)x$ and grows like a
  $\cosh$; nothing is jerked, so energy and momentum agree.
- **Act II:** the piling chain makes the floor read *three times* the landed
  weight, because stopping the arriving links costs an extra $2\lambda y g$;
  energy is plainly dissipated.
- **Act III:** the folded chain's free end falls *faster than $g$*; whether
  energy is conserved hinges on how the fold negotiates the links, and the two
  idealisations differ by exactly $\tfrac14\lambda g x^2$.

And **Act IV** ran the bookkeeping in reverse — mass shed instead of gathered —
and got Tsiolkovsky's $\Delta v = v_e\ln(m_0/m_1)$ to nine digits, the
$1766\ \mathrm{m\,s^{-1}}$ gravity loss of a vertical burn, and the staging
argument: with $10\%$ structure, orbit is unreachable in one piece ($6649$ of
$9000\ \mathrm{m\,s^{-1}}$) and comfortable in two ($9964$).

The single thread is {eq}`eq-dpdt`: $\mathbf F_\text{ext}=d\mathbf p/dt$ for the
whole system, applied with the momentum carried across the moving boundary, not
$F=ma$ on a mass that is quietly changing.

## Outlook

- **The chain fountain (Mould effect):** a chain that leaps *above* its beaker as
  it pours out: a modern variable-mass puzzle with a momentum-flux explanation.
- **Beyond Exercise 6's rocket**: real ascent trajectories add drag, pitch
  programs, and the gravity turn — the $d\mathbf p/dt$ bookkeeping is the same,
  wrapped in an optimal-control problem.
- **The catenary**, the *static* hanging chain, is a problem in the calculus of
  variations: the tool of [§2.8](../02-classical-mechanics/brachistochrone.ipynb).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()